In [1]:
#!pip install albumentations

In [7]:
### Augmentation uniquement pour les répertoires Train
import os
import cv2
import random
import shutil
import albumentations as A

# --- Pipeline d'augmentation ---

#Augmentations légères
augment = A.Compose([
    A.Rotate(limit=10, p=0.8),  # rotation très légère ±10°
    A.ShiftScaleRotate(
        shift_limit=0.05,       # translation légère
        scale_limit=0.05,       # zoom léger
        rotate_limit=10,        # rotation max 10°
        p=0.8
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),  # flou léger
    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.3
    ),
    A.HorizontalFlip(p=0.3),   # flip horizontal léger
])


### Répertoires
src_root = "C:\\Users\\fabbg\\Documents\\Projet DS\\Resized_128\\Data_Projet_DS"
dst_root = "C:\\Users\\fabbg\\Documents\\Projet DS\\mes_images_Train_augmentees"

# --- Copier toute l'arborescence (sans augmentation) ---
shutil.copytree(src_root, dst_root, dirs_exist_ok=True)

# --- Trouver tous les dossiers train contenant des images ---
train_folders = []
for root, dirs, files in os.walk(src_root):
    if "train" in root.lower():   # <--- train détecté dans tout le chemin
        imgs = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        if imgs:
            train_folders.append((root, imgs))


# --- Calcul du nombre maximum dans les dossiers train ---
counts = {root: len(imgs) for root, imgs in train_folders}
max_count = max(counts.values())
print("Nombre cible dans les dossiers train :", max_count)

# --- Augmentation uniquement dans les dossiers train ---
for src_path, imgs in train_folders:
    dst_path = src_path.replace(src_root, dst_root)

    count = len(imgs)
    to_generate = max_count - count

    print(f"{src_path} : {count} images → génération de {to_generate} images")

    for i in range(to_generate):
        img_name = random.choice(imgs)
        img_path = os.path.join(src_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        augmented = augment(image=img)["image"]

        out_name = f"aug_{i}_{img_name}"
        out_path = os.path.join(dst_path, out_name)

        cv2.imwrite(out_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

print("Nouvelle arborescence générée (augmentation uniquement dans train).")


Nombre cible dans les dossiers train : 391
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\bottle\train\good : 209 images → génération de 182 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\cable\train\good : 224 images → génération de 167 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\capsule\train\good : 219 images → génération de 172 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\carpet\train\good : 280 images → génération de 111 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\grid\train\good : 264 images → génération de 127 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\hazelnut\train\good : 391 images → génération de 0 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\leather\train\good : 245 images → génération de 146 images
C:\Users\fabbg\Documents\Projet DS\Resized_128\Data_Projet_DS\metal_nut\train\good : 220 images → génération de 171 im

In [10]:
import os
import shutil

# Arborescence déjà générée
src_root = "C:\\Users\\fabbg\\Documents\\Projet DS\\mes_images_Train_augmentees"

# Nouvelle arborescence globale
dst_root = "C:\\Users\\fabbg\\Documents\\Projet DS\\NouvelleArbo"
train_root = os.path.join(dst_root, "Train")
test_root = os.path.join(dst_root, "Test")

os.makedirs(train_root, exist_ok=True)
os.makedirs(test_root, exist_ok=True)

# Parcours des classes
for classe in os.listdir(src_root):
    classe_path = os.path.join(src_root, classe)
    if not os.path.isdir(classe_path):
        continue

    # --- Partie Train ---
    train_src = os.path.join(classe_path, "Train")
    if os.path.isdir(train_src):
        # Sous-répertoire good
        good_src = os.path.join(train_src, "good")
        if os.path.isdir(good_src):
            dst_good = os.path.join(train_root, classe, "good")
            os.makedirs(dst_good, exist_ok=True)

            # Copier toutes les images good
            for img in os.listdir(good_src):
                if img.lower().endswith((".png", ".jpg", ".jpeg")):
                    shutil.copy(os.path.join(good_src, img),
                                os.path.join(dst_good, img))

    # --- Partie Test ---
    test_src = os.path.join(classe_path, "Test")
    if os.path.isdir(test_src):
        for defect in os.listdir(test_src):
            defect_src = os.path.join(test_src, defect)
            if os.path.isdir(defect_src):
                dst_defect = os.path.join(test_root, classe, defect)
                os.makedirs(dst_defect, exist_ok=True)

                # Copier les images du défaut
                for img in os.listdir(defect_src):
                    if img.lower().endswith((".png", ".jpg", ".jpeg")):
                        shutil.copy(os.path.join(defect_src, img),
                                    os.path.join(dst_defect, img))

print("Nouvelle arborescence Train/Test créée avec succès.")


Nouvelle arborescence Train/Test créée avec succès.
